# 27. Motion Context + Feature Extraction Test (Pipeline 08-09)

- Goal: inspect motion/role context handoff and feature extraction outputs in one stage-check surface.
- Docs: `docs_eng/pipeline/08_motion_attribution.md`, `docs_eng/pipeline/09_feature_extraction.md` / Korean mirrors.
- Prerequisite: 20-26 stage checks should already pass.
- Input: normalized, segmented pose dataframe for the selected real sample.
- Output: attribution provenance, `FeatureRecord` list, feature dataframe, feature-context summary, and pipeline integration report.
- Checks: motion-context output contract, feature context, FeatureRecord fields, rep/phase records, source fields, availability, and ⑧/⑨ integration readiness.
- Policy: this stage computes context, features, and provenance only. It does not modify coordinates, relabel reps/phases, or score movement quality.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from movement.config import LANDMARKS
from movement.features import (
    FeatureContext,
    FeatureRecord,
    extract_rep_features,
    features_to_dataframe,
    resolve_feature_context,
    summarize_phase_to_rep,
)
from movement.motion_attribution import (
    AttributionReport,
    AttributionThresholds,
    attribute_motion,
)
from movement.pipeline import (
    AnnotationConfig,
    ExerciseDefinitionConfig,
    FeaturesConfig,
    MotionAttributionConfig,
    NormalizationConfig,
    PhaseSegmentationConfig,
    PipelineConfig,
    PreprocessingConfig,
    RepSegmentationConfig,
    ValidationConfig,
    run_pipeline,
)
from movement.segmentation import segment_phases, segment_reps
from movement.stage_context import prepare_previous_stage_inputs

print('imports OK')

## Data Setup

Prepare the selected real sample through ⑧ Motion Attribution. The setup mirrors the pipeline order while keeping this notebook focused on the combined ⑧/⑨ handoff.

In [ ]:
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / 'pyproject.toml').exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError('Could not find project root containing pyproject.toml')
    PROJECT_ROOT = PROJECT_ROOT.parent

pose_csv = 'data/pose/mediapipe/no_consent/20260517/p01_squat_set1_output_pose.csv'
annotation_csv = 'data/pose/mediapipe/no_consent/20260517/p01_squat_set1_annotation.csv'
TARGET_EXERCISE_ID = 'draft_squat'

pre_config = PreprocessingConfig(enabled=True)
norm_config = NormalizationConfig(
    enabled=True,
    keep_reference_columns=True,
    model_depth_scale=1.0,
)
stage_inputs = prepare_previous_stage_inputs(
    prepare_until='normalization',
    pose_csv=pose_csv,
    annotation_csv=annotation_csv,
    exercise_id=TARGET_EXERCISE_ID,
    landmarks=LANDMARKS,
    preprocessing_config=pre_config,
    normalization_config=norm_config,
)

df_raw = stage_inputs.raw_df
ann_df = stage_inputs.annotation_df
exercise_def = stage_inputs.exercise_definition
norm_df = stage_inputs.normalized_df
annotation_path = stage_inputs.annotation_csv
TARGET_DEFINITIONS_DIR = stage_inputs.definitions_dir

df_rep_seg, rep_report = segment_reps(norm_df, exercise_def, fps_default=30.0)
df_seg, phase_reports = segment_phases(df_rep_seg, exercise_def, fps_default=30.0)
thresholds = AttributionThresholds(active=0.70, ambiguous=0.55, swap=0.85)
df_attr, attr_report = attribute_motion(
    df_seg,
    exercise_def,
    thresholds=thresholds,
    mode='conservative',
)
attr_dict = attr_report.as_dict()
feature_context = resolve_feature_context(df_attr, exercise_def, attr_dict)

setup_summary = pd.DataFrame([
    {'item': 'exercise_id', 'value': exercise_def.exercise_id},
    {'item': 'laterality', 'value': exercise_def.classification.get('laterality')},
    {'item': 'frames_loaded', 'value': len(df_raw)},
    {'item': 'rep_frames', 'value': int(df_attr['segment_type'].eq('rep').sum())},
    {'item': 'rep_count', 'value': int(df_attr['rep_id'].dropna().nunique())},
    {'item': 'phase_labels', 'value': ', '.join(map(str, sorted(df_attr['phase'].dropna().unique())))},
    {'item': 'motion_attribution_skipped', 'value': attr_dict['skipped']},
    {'item': 'motion_attribution_skip_reason', 'value': attr_dict['skip_reason']},
    {'item': 'feature_context_role_mode', 'value': feature_context.role_mode},
    {'item': 'definitions_dir', 'value': str(TARGET_DEFINITIONS_DIR)},
])
display(setup_summary)

## Feature Context Resolution

For `draft_squat`, active-side attribution should stay skipped, while ⑨ receives bilateral symmetry / side-bias context for feature records that need it.

In [ ]:
assert isinstance(feature_context, FeatureContext)
context_df = pd.DataFrame([feature_context.as_dict()])
display(context_df.T.rename(columns={0: 'value'}))

assert feature_context.laterality == exercise_def.classification.get('laterality')
assert feature_context.role_mode == 'bilateral_symmetry'
assert feature_context.attribution_confidence == 'not_assessed'
assert 'active_side_attribution_not_applicable' in feature_context.context_reasons
assert attr_dict['skipped'] is True
print('PASS: bilateral sample resolves to symmetry context, not active-side attribution')

## Motion Context Contract

This compact check replaces the former standalone 27 notebook for the default follow-along path. It verifies that motion attribution may add provenance columns, but does not mutate coordinates, `rep_id`, or `phase`.

In [ ]:
ATTRIBUTION_COLUMNS = [
    'detected_active_limb',
    'expected_active_limb',
    'attribution_consistent',
    'attribution_confidence',
    'attribution_action',
]

assert isinstance(attr_report, AttributionReport)
assert attr_dict['exercise_id'] == TARGET_EXERCISE_ID
assert attr_dict['laterality'] == exercise_def.classification.get('laterality')
assert attr_dict['execution_pattern'] == 'bilateral'
assert attr_dict['performance_side_sequence']['mode'] == 'none'
assert attr_dict['thresholds']
assert attr_dict['skipped'] is True
assert attr_dict['skip_reason']

for col in ATTRIBUTION_COLUMNS:
    assert col in df_attr.columns, f'missing attribution column: {col}'

null_counts = {col: int(df_attr[col].notna().sum()) for col in ATTRIBUTION_COLUMNS}
display(pd.Series(null_counts, name='non_null_frames').to_frame())
assert all(count == 0 for count in null_counts.values())

coord_cols = [c for c in df_seg.columns if c.endswith(('_norm_x', '_norm_y', '_norm_z'))]
for col in coord_cols:
    assert np.allclose(df_attr[col].to_numpy(), df_seg[col].to_numpy(), equal_nan=True), f'{col} was modified'

pd.testing.assert_series_equal(df_attr['rep_id'], df_seg['rep_id'], check_names=False)
pd.testing.assert_series_equal(df_attr['phase'], df_seg['phase'], check_names=False)
print(f'PASS: motion context columns present; coordinates/rep_id/phase unchanged ({len(coord_cols)} norm coordinate columns checked)')

## Direct Feature Extraction

Run the current `extract_rep_features()` implementation. Feature values are computed first, then the ⑨ feature-context application substep attaches `role_context` to the feature families that need it.

In [ ]:
records = extract_rep_features(df_attr, exercise_def)
summary_records = summarize_phase_to_rep(records)
all_records = records + summary_records

rep_records = [r for r in records if r.phase is None]
phase_records = [r for r in records if r.phase is not None]
df_feat = features_to_dataframe(all_records)

feature_counts = (
    df_feat.assign(feature_family=df_feat['feature_id'].str.split('.').str[:2].str.join('.'))
    .groupby(['feature_family', 'phase'], dropna=False)
    .size()
    .rename('records')
    .reset_index()
)
display(feature_counts)
print(f'total records       : {len(records)}')
print(f'summary records     : {len(summary_records)}')
print(f'rep-level records   : {len(rep_records)}')
print(f'phase-level records : {len(phase_records)}')

## Check 1: FeatureRecord Contract

In [ ]:
for record in all_records:
    assert isinstance(record, FeatureRecord)
    assert record.feature_id, f'feature_id empty: {record}'
    assert record.exercise_id == TARGET_EXERCISE_ID, f'wrong exercise_id: {record.exercise_id}'
    assert record.value is not None, f'value None: {record}'
    assert record.unit, f'unit empty: {record}'
    assert record.source_fields, f'source_fields empty: {record.feature_id}'
print(f'PASS: all {len(all_records)} FeatureRecord fields valid')

## Check 2: Rep And Phase Coverage

In [ ]:
assert rep_records, 'no rep-level records produced'
rep_ids_with_features = {r.rep_id for r in rep_records if r.rep_id is not None}
all_rep_ids = set(df_attr.loc[df_attr['segment_type'] == 'rep', 'rep_id'].dropna().unique())
missing_rep_ids = all_rep_ids - rep_ids_with_features
print(f'reps in data       : {sorted(all_rep_ids)}')
print(f'reps with features : {sorted(rep_ids_with_features)}')
assert not missing_rep_ids, f'reps without features: {missing_rep_ids}'

if phase_records:
    phase_labels = {r.phase for r in phase_records}
    print(f'phase labels in feature records: {sorted(phase_labels)}')
    assert phase_labels.issubset(set(df_attr['phase'].dropna().unique()))
    print('PASS: phase-level feature records are present and use existing phase labels')
else:
    print('NOTE: no phase-level records emitted for this run')

## Check 3: Source Fields And Phase Provenance

In [ ]:
phase_with_provenance = [
    r for r in phase_records
    if any('phase_segmentation' in source for source in r.source_fields)
]
if phase_records:
    assert len(phase_with_provenance) == len(phase_records)
print(f'phase records with phase_segmentation provenance: {len(phase_with_provenance)} / {len(phase_records)}')

source_summary = (
    df_feat.assign(source_field=df_feat['source_fields'].str.split('|'))
    .explode('source_field')
    .groupby('source_field')
    .size()
    .rename('records')
    .sort_values(ascending=False)
    .reset_index()
)
display(source_summary.head(12))
print('PASS: source_fields are present and phase records preserve segmentation provenance')

## Check 4: Availability And View Reliability

For p01 `draft_squat`, symmetry features are computable but may be low confidence because monocular depth evidence is weak. This belongs to feature availability, not movement-quality scoring.

In [ ]:
availability_summary = (
    df_feat.assign(feature_family=df_feat['feature_id'].str.split('.').str[:2].str.join('.'))
    .groupby(['feature_family', 'availability', 'view_reliability', 'depth_dependency'], dropna=False)
    .size()
    .rename('records')
    .reset_index()
)
display(availability_summary)

symmetry_rows = df_feat[df_feat['feature_id'].str.startswith('spatial.symmetry.')]
if not symmetry_rows.empty:
    display(symmetry_rows[['feature_id', 'rep_id', 'value', 'availability', 'availability_reasons', 'depth_dependency', 'model_depth_reliability']].head(12))
    assert set(symmetry_rows['availability']).issubset({'assessed', 'low_confidence', 'not_assessed'})
print('PASS: availability metadata is attached without changing feature values into scores')

## Check 5: ⑧/⑨ Merge-Readiness Inspection

This cell answers the current merge question. For `draft_squat`, bilateral symmetry features should now carry context, while active-side attribution remains skipped.

In [ ]:
def context_need(feature_id):
    if feature_id.startswith('spatial.symmetry.'):
        return 'bilateral_symmetry_context'
    if feature_id.startswith('control.compensation.'):
        return 'candidate_specific_context'
    if feature_id.startswith('spatial.rom.'):
        return 'joint_angle_context'
    if feature_id.startswith('temporal.'):
        return 'timing_context'
    return 'general_feature_context'

context_usage_df = df_feat.assign(
    context_need=df_feat['feature_id'].map(context_need),
    has_role_context=df_feat['role_context'].map(lambda value: isinstance(value, dict) and bool(value)),
    has_motion_attribution_source=df_feat['source_fields'].str.contains('motion_attribution', na=False),
)
context_usage_summary = (
    context_usage_df.groupby(['context_need', 'has_role_context', 'has_motion_attribution_source'])
    .size()
    .rename('records')
    .reset_index()
)
display(context_usage_summary)

symmetry_context_rows = context_usage_df.loc[context_usage_df['context_need'].eq('bilateral_symmetry_context')]
needs_bilateral_context = not symmetry_context_rows.empty
symmetry_context_attached = bool(needs_bilateral_context and symmetry_context_rows['has_role_context'].all())
motion_source_attached = context_usage_df['has_motion_attribution_source'].any()
merge_readiness = {
    'feature_context_role_mode': feature_context.role_mode,
    'needs_bilateral_context': bool(needs_bilateral_context),
    'symmetry_context_attached_to_records': bool(symmetry_context_attached),
    'motion_attribution_source_attached': bool(motion_source_attached),
    'recommended_decision': (
        'context_application_ready_keep_27_as_qc_until_broader_exercise_review'
        if symmetry_context_attached
        else 'add_09_context_application_substep'
    ),
}
display(pd.DataFrame([merge_readiness]).T.rename(columns={0: 'value'}))
assert bool(needs_bilateral_context) is True
assert bool(symmetry_context_attached) is True
assert bool(motion_source_attached) is False
print('PASS: ⑨ applies bilateral symmetry context while ⑧ active-side attribution remains skipped')

## Check 6: features_to_dataframe() Contract

In [ ]:
required_cols = [
    'feature_id',
    'exercise_id',
    'rep_id',
    'value',
    'unit',
    'source_fields',
    'phase',
    'availability',
    'role_context',
]
for col in required_cols:
    assert col in df_feat.columns, f'missing column: {col}'
print(f'PASS: features_to_dataframe() shape={df_feat.shape}')
display(df_feat[['feature_id', 'rep_id', 'phase', 'value', 'unit', 'availability', 'role_context']].head(12))

## Check 7: Pipeline Integration

In [ ]:
cfg = PipelineConfig()
cfg.validation = ValidationConfig(enabled=True)
cfg.annotation = AnnotationConfig(enabled=True, path=annotation_path)
cfg.exercise_definition = ExerciseDefinitionConfig(
    enabled=True,
    definitions_dir=str(TARGET_DEFINITIONS_DIR),
    exercise_id=TARGET_EXERCISE_ID,
)
cfg.preprocessing = pre_config
cfg.normalization = norm_config
cfg.rep_segmentation = RepSegmentationConfig(enabled=True, fps_default=30.0)
cfg.phase_segmentation = PhaseSegmentationConfig(enabled=True, fps_default=30.0)
cfg.motion_attribution = MotionAttributionConfig(enabled=True)
cfg.features = FeaturesConfig(enabled=True)

pipe_df, pipe_report = run_pipeline(df_raw, config=cfg, landmarks=LANDMARKS, ann_df=ann_df)

assert 'motion_attribution' in pipe_report
assert 'features' in pipe_report
pipe_features = pd.DataFrame(pipe_report['features'])
pipe_symmetry = pipe_features[pipe_features['feature_id'].str.startswith('spatial.symmetry.')]
pipe_symmetry_has_context = pipe_symmetry['role_context'].map(lambda value: isinstance(value, dict) and bool(value))
print('pipeline ⑧ skipped: {}'.format(pipe_report['motion_attribution']['skipped']))
print('pipeline ⑨ feature records: {}'.format(len(pipe_features)))
print('pipeline symmetry records with context: {} / {}'.format(int(pipe_symmetry_has_context.sum()), len(pipe_symmetry)))
print('steps executed: {}'.format(list(pipe_report.keys())))
assert pipe_report['motion_attribution']['skipped'] is True
assert len(pipe_features) == len(records)
assert not pipe_symmetry.empty
assert pipe_symmetry_has_context.all()
print('PASS: pipeline integration preserves separate ⑧ QC and applies ⑨ feature context')

## Check Summary

This notebook is the combined follow-along checkpoint for ⑧/⑨. Current p01 `draft_squat` produces bilateral-symmetry context, skips active-side attribution, verifies motion-context non-mutation, and attaches `role_context` to `spatial.symmetry.*` feature records without changing coordinates, rep/phase labels, feature values, availability, or scores. The `attribute_motion()` helper remains in code for later alternating/unilateral review, but the default stage-check path no longer needs a separate motion-attribution notebook.